<a href="https://colab.research.google.com/github/farrelrassya/LLM-Design-Patterns-Building-Robust-and-Efficient-AI-Systems/blob/main/21.Tree_of_Thoughts_Prompting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bab 21 — Tree-of-Thoughts Prompting

*LLM Design Patterns* · notebook pendamping, edisi Colab GPU

Ambil soal *Game of 24*: dari angka 4, 9, 10, dan 13, susun + − × ÷ sehingga hasilnya 24, dengan setiap angka dipakai tepat sekali. Pola *chain of thought* (CoT) dari bab sebelumnya menempuh satu rantai langkah saja, sehingga langkah pertama yang keliru tidak bisa dibatalkan: setelah 4 + 9 = 13, sisa 10, 13, dan 13 tidak mungkin lagi menjadi 24.

*Tree of thoughts* (ToT) memperlakukan penalaran sebagai pencarian: di setiap langkah model membangkitkan beberapa kandidat langkah (*pikiran*), sebuah penilai memberi skor, cabang yang lemah dipangkas, dan pencarian boleh mundur bila buntu. Di makalah aslinya (Yao dkk. 2023), GPT-4 menyelesaikan 4% soal Game of 24 dengan CoT dan 74% dengan ToT (dari literatur), dengan GPT-4 sendiri sebagai pembangkit sekaligus penilai.

Notebook ini mengikuti urutan bab dengan `gpt2-large` (774 juta parameter) seperti di buku, model yang jauh lebih kecil daripada GPT-4. Pertanyaan yang terus kita bawa: **kapan cabang dan penilai benar-benar menolong, dan kapan pohon hanya memperbanyak teks?**

> **Catatan runtime.** Jalankan di Colab dengan GPU (*Runtime > Change runtime type > T4 GPU*). Sel setup menguji CUDA sebelum memuat model dan berhenti dengan petunjuk bila GPU tidak ada. Mode CPU hanya jalan bila Anda sendiri mengubah `ALLOW_CPU` menjadi `True`, dengan GPT-2 124M dan ukuran eksperimen yang diperkecil.
>
> Notebook ini belum pernah dijalankan di GPU atau Colab, jadi belum ada output; yang sudah dijalankan hanya mode CPU kecil di Mac Apple Silicon (detailnya di `CHANGES_AND_VALIDATION.md`). Angka di narasi ada tiga jenis: hasil yang tidak bergantung pada model, angka makalah (*dari literatur*), dan angka *run sumber*, yaitu output tersimpan edisi sebelumnya notebook ini (GPT-2 124M, CPU 2 vCPU, ukuran eksperimen hampir sama dengan mode GPU). Run sumber hanya pembanding; hasil Anda akan berbeda.

## Pohon pikiran tanpa model

Kita lihat dulu mekanismenya pada Game of 24, dengan cabang dari enumerasi biasa, bukan dari model, sehingga hasilnya pasti. Istilah yang dipakai sepanjang bab:

- **State**: angka yang tersisa plus langkah yang sudah diambil, mula-mula (4, 9, 10, 13).
- **Pikiran** atau **cabang**: satu langkah berikutnya, misalnya `13 - 9 = 4`, yang menghasilkan state (4, 10, 4). Setiap state di pohon adalah **simpul**; simpul terbawah adalah **daun**.
- **Penilai**: memberi label *sure*, *maybe*, atau *impossible*, seperti di makalah Yao dkk. Untuk state dua angka ia memeriksa secara pasti apakah 24 masih tercapai; untuk tiga atau empat angka ia sengaja menjawab *maybe*.
- **Pangkas dan berhenti**: cabang *impossible* dibuang; pencarian berhenti saat menemukan 24 atau saat cabang habis.

Sel berikut menjalankan DFS (menyelami satu cabang sampai habis sebelum mundur) dan BFS (menyelesaikan satu tingkat sebelum turun), dengan dan tanpa penilai. Biayanya dihitung dalam *ekspansi*, yaitu berapa kali sebuah state diperluas menjadi cabang-cabangnya; pada ToT dengan LLM, satu ekspansi adalah satu panggilan `generate`. Kerja penilai ikut dihitung, karena menilai state dua angka a dan b sama dengan satu ekspansi tersembunyi: keenam hasilnya (a + b, a × b, a − b, b − a, a ÷ b, b ÷ a) dicoba satu per satu. Sel ini hanya memakai CPU.

In [1]:
from collections import deque
from fractions import Fraction
from itertools import combinations


def expand(state):
    """Semua pikiran berikutnya: ambil dua angka, ganti dengan hasil satu operasi."""
    nums, steps = state
    children = []
    for i, j in combinations(range(len(nums)), 2):
        a, b = nums[i], nums[j]
        rest = [nums[k] for k in range(len(nums)) if k not in (i, j)]
        options = [(a + b, f"{a} + {b}"), (a * b, f"{a} * {b}"), (a - b, f"{a} - {b}"), (b - a, f"{b} - {a}")]
        if b != 0:
            options.append((a / b, f"{a} / {b}"))
        if a != 0:
            options.append((b / a, f"{b} / {a}"))
        for value, text in options:
            children.append((tuple(rest + [value]), steps + [f"{text} = {value}"]))
    return children


def evaluate(state):
    """Penilai ala Yao dkk.: sure / maybe / impossible."""
    nums, _ = state
    if len(nums) == 1:
        return "sure" if nums[0] == 24 else "impossible"
    if len(nums) == 2:
        return "sure" if any(child[0][0] == 24 for child in expand(state)) else "impossible"
    return "maybe"


def search(numbers, strategy, use_evaluator):
    frontier = deque([(tuple(Fraction(n) for n in numbers), [])])
    expansions = 0
    lookaheads = 0                                   # ekspansi tersembunyi di dalam penilai
    while frontier:
        state = frontier.pop() if strategy == "DFS" else frontier.popleft()   # tumpukan vs antrean
        if len(state[0]) == 1:
            if state[0][0] == 24:
                return state[1], expansions, lookaheads    # berhenti: solusi ditemukan
            continue
        expansions += 1
        children = expand(state)
        if use_evaluator:
            lookaheads += sum(len(c[0]) == 2 for c in children)              # evaluate() memperluas state dua angka
            children = [c for c in children if evaluate(c) != "impossible"]   # pangkas
        frontier.extend(reversed(children) if strategy == "DFS" else children)
    return None, expansions, lookaheads              # berhenti: tidak ada cabang tersisa


for numbers in ([4, 9, 10, 13], [1, 1, 1, 1]):
    for strategy in ("DFS", "BFS"):
        for use_evaluator in (False, True):
            path, n, look = search(numbers, strategy, use_evaluator)
            label = "dengan penilai" if use_evaluator else "tanpa penilai"
            found = "; ".join(path) if path else "tidak ada solusi"
            print(f"{str(numbers):<15} {strategy} {label:<15} ekspansi {n:>4} + di dalam penilai {look:>4} "
                  f"= {n + look:>4} -> {found}")

[4, 9, 10, 13]  DFS tanpa penilai   ekspansi  157 + di dalam penilai    0 =  157 -> 4 - 10 = -6; 9 - 13 = -4; -6 * -4 = 24
[4, 9, 10, 13]  DFS dengan penilai  ekspansi   11 + di dalam penilai  162 =  173 -> 4 - 10 = -6; 9 - 13 = -4; -6 * -4 = 24
[4, 9, 10, 13]  BFS tanpa penilai   ekspansi  685 + di dalam penilai    0 =  685 -> 4 - 10 = -6; 9 - 13 = -4; -6 * -4 = 24
[4, 9, 10, 13]  BFS dengan penilai  ekspansi   46 + di dalam penilai  648 =  694 -> 4 - 10 = -6; 9 - 13 = -4; -6 * -4 = 24
[1, 1, 1, 1]    DFS tanpa penilai   ekspansi  661 + di dalam penilai    0 =  661 -> tidak ada solusi
[1, 1, 1, 1]    DFS dengan penilai  ekspansi   37 + di dalam penilai  624 =  661 -> tidak ada solusi
[1, 1, 1, 1]    BFS tanpa penilai   ekspansi  661 + di dalam penilai    0 =  661 -> tidak ada solusi
[1, 1, 1, 1]    BFS dengan penilai  ekspansi   37 + di dalam penilai  624 =  661 -> tidak ada solusi


Keempat konfigurasi menemukan (4 − 10) × (9 − 13) = 24. Tanpa penilai, DFS sampai di sana setelah 157 ekspansi dan BFS setelah 685, yaitu seluruh pohon di atas daun (1 + 36 + 648), karena semua solusi ada di tingkat terakhir.

Dengan penilai, pencarian sendiri hanya memperluas 11 state (DFS) dan 46 (BFS). Tetapi kerjanya pindah, bukan hilang: pada DFS, penilai mengerjakan 162 ekspansi tersembunyi untuk menilai state dua angka, sehingga totalnya naik dari 157 menjadi 173, dan pada BFS dari 685 menjadi 694. Penilai ini sempurna justru karena diam-diam mengerjakan langkah terakhir pencarian lebih dulu. Pada (1, 1, 1, 1), yang tidak punya solusi, totalnya 661 dengan atau tanpa penilai.

Itulah pelajaran pertama: pemangkasan baru menghemat bila menilai jauh lebih murah daripada ekspansi yang dicegahnya. Pada ToT dengan LLM, menilai biasanya juga memanggil model; di makalah Yao dkk., setiap kandidat pikiran dinilai GPT-4 tiga kali. Penilai buku yang akan kita temui justru sangat murah, karena hanya menghitung kata. Seberapa dekat ia dengan pemeriksa setepat ini?

## Menyiapkan GPU

Sel berikut memastikan CUDA benar-benar bisa dipakai, bukan sekadar terdeteksi: ia mengalikan dua matriks kecil di GPU dan menunggu hasilnya (`synchronize`). Lalu ia memilih presisi: bfloat16 untuk GPU dengan compute capability 8 ke atas, float16 untuk T4 (7.5). `torch.cuda.is_bf16_supported()` sengaja tidak dipakai, karena di T4 ia bisa menjawab `True` lewat emulasi. Pengaturan ukuran eksperimen juga ada di sini.

In [2]:
import concurrent.futures
import importlib.util
import itertools
import os
import random
import re
import subprocess
import sys
import time
from collections import Counter

if importlib.util.find_spec("transformers") is None:      # Colab sudah memasangnya; ini untuk runtime lain
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.44"])

import numpy as np
import torch
import transformers

ALLOW_CPU = os.environ.get("TOT_ALLOW_CPU", "0") == "1"   # ubah menjadi True hanya untuk demo CPU kecil
GPU_HELP = "Aktifkan GPU: Runtime > Change runtime type > Hardware accelerator: GPU (mis. T4), lalu jalankan ulang dari atas."

if torch.cuda.is_available():
    try:
        probe = torch.randn(1024, 1024, device="cuda")
        product = probe @ probe
        torch.cuda.synchronize()
        assert torch.isfinite(product).all()
    except (RuntimeError, AssertionError) as err:
        raise RuntimeError(f"CUDA terdeteksi tetapi gagal menjalankan perhitungan kecil: {err!r}\n{GPU_HELP}") from err
    DEVICE = "cuda"
    major, minor = torch.cuda.get_device_capability(0)
    DTYPE = torch.bfloat16 if major >= 8 else torch.float16
    vram_free, vram_total = torch.cuda.mem_get_info()
    print(f"GPU: {torch.cuda.get_device_name(0)} | compute capability {major}.{minor} | "
          f"VRAM bebas {vram_free / 2**30:.1f} dari {vram_total / 2**30:.1f} GiB")
    del probe, product
elif ALLOW_CPU:
    DEVICE = "cpu"
    DTYPE = torch.float32
    print("PERHATIAN: CUDA tidak ada dan mode CPU dipilih secara eksplisit. "
          "Model diganti GPT-2 124M dan ukuran eksperimen diperkecil.")
else:
    raise RuntimeError(f"CUDA tidak tersedia. {GPU_HELP}\n"
                       "Untuk demo CPU kecil, set ALLOW_CPU = True di sel ini (lebih lambat, ukuran diperkecil).")

SMALL = DEVICE == "cpu"
MODEL_NAME = os.environ.get("TOT_MODEL") or ("openai-community/gpt2" if SMALL else "openai-community/gpt2-large")
NEW_TOKENS = 30 if SMALL else 100      # token baru per ekspansi di fungsi-fungsi buku
UTH_TOKENS = 24 if SMALL else 64       # token per cabang di Under the hood 1 dan 4
N_ITEMS = 6 if SMALL else 40           # soal per tugas di Under the hood 3
K_CAND = 8                             # kandidat per soal di Under the hood 3
MS_DEPTH = 2 if SMALL else 3           # kedalaman pohon per langkah di soal bertahap
DYN_DEPTH = 1 if SMALL else 2          # kedalaman dynamic_tot yang dijalankan (bawaan buku: 5)
CARRY_TOKENS = 150                     # konteks dari langkah sebelumnya yang dibawa di soal bertahap
print(f"torch {torch.__version__} | transformers {transformers.__version__} | perangkat {DEVICE} | {DTYPE} | {MODEL_NAME}")

GPU: Tesla T4 | compute capability 7.5 | VRAM bebas 14.4 dari 14.6 GiB
torch 2.11.0+cu130 | transformers 5.20.2 | perangkat cuda | torch.float16 | openai-community/gpt2-large


Baris yang tercetak adalah bukti pertama bahwa GPU berfungsi; di T4 Anda akan melihat compute capability 7.5 dan `torch.float16`.

Berikutnya `gpt2-large` dimuat (`openai-community/gpt2-large` di Hugging Face Hub, model yang sama dengan `"gpt2-large"` di buku), dipindah ke GPU, dan dikonversi ke presisi tadi. GPT-2 tidak punya token *padding*, token pengisi yang menyamakan panjang prompt dalam satu batch, jadi token akhir teks (`eos`) dipakai sebagai gantinya. Padding diletakkan di kiri supaya semua prompt berakhir di posisi yang sama dan bisa dilanjutkan sekaligus.

Kita belum tahu pasti apakah GPT-2 stabil di float16, jadi sel ini menjalankan model pada satu prompt pendek dan satu prompt hampir 1.000 token, lalu memeriksa bahwa semua *logit*-nya (skor mentah untuk setiap kemungkinan token berikutnya) terhingga. Bila tidak, model pindah ke float32 di GPU yang sama, dan hal itu dicetak.

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

transformers.logging.set_verbosity_error()
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token      # GPT-2 tidak punya token padding
tokenizer.padding_side = "left"                # batch: semua prompt berakhir di posisi yang sama
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model = model.to(DEVICE, dtype=DTYPE).eval()
CONTEXT = getattr(model.config, "n_positions", None) or model.config.max_position_embeddings


@torch.no_grad()
def logits_are_finite():
    sentence = tokenizer(" Tree of thoughts explores several branches, scores them, and prunes the weak ones.")["input_ids"]
    long_ids = (sentence * (CONTEXT // len(sentence) + 1))[:CONTEXT - 64]   # hampir sepanjang konteks model
    batch = tokenizer(["Tree of thoughts explores", tokenizer.decode(long_ids)],
                      return_tensors="pt", padding=True).to(DEVICE)
    logits = model(**batch).logits
    return bool(torch.isfinite(logits[batch["attention_mask"].bool()]).all())


if not logits_are_finite():
    print(f"{DTYPE} menghasilkan logit tak terhingga; model dipindah ke float32 di {DEVICE}.")
    DTYPE = torch.float32
    model = model.to(dtype=DTYPE)
    assert logits_are_finite(), "Logit tetap tak terhingga di float32"

assert all(p.device.type == DEVICE for p in model.parameters())
print(f"{MODEL_NAME}: {sum(p.numel() for p in model.parameters()) / 1e6:.0f} juta parameter | konteks {CONTEXT} token")
print(f"parameter di {sorted({str(p.device) for p in model.parameters()})} | dtype {next(model.parameters()).dtype}")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.25GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/436 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

openai-community/gpt2-large: 774 juta parameter | konteks 1024 token
parameter di ['cuda:0'] | dtype torch.float16


Tiga fungsi bantu dipakai berulang kali. `generate_batch` melanjutkan banyak prompt dalam satu panggilan `generate`, `k` lanjutan per prompt. `mean_logprob` menghitung *log-probabilitas* rata-rata per token sebuah lanjutan, yaitu peluang yang diberikan model pada setiap tokennya dalam skala logaritma. Nilainya negatif; makin dekat ke 0, makin wajar teks itu menurut model (−0,7 kira-kira peluang 50% per token). Proyeksi ke kosakata dan `log_softmax`-nya dihitung di float32 supaya pembulatan float16/bfloat16 tidak menggeser urutan kandidat. `sync` menunggu GPU selesai sebelum stopwatch dibaca.

In [4]:
def sync():
    if DEVICE == "cuda":
        torch.cuda.synchronize()


@torch.no_grad()
def generate_batch(prompts, k, new_tokens, **decoding):
    """Satu panggilan generate untuk banyak prompt; mengembalikan k lanjutan per prompt."""
    enc = tokenizer(prompts, return_tensors="pt", padding=True).to(DEVICE)
    out = model.generate(**enc, max_new_tokens=new_tokens, num_return_sequences=k,
                         pad_token_id=tokenizer.eos_token_id, **decoding)
    texts = tokenizer.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return [texts[i * k:(i + 1) * k] for i in range(len(prompts))]


@torch.no_grad()
def mean_logprob(prompts, continuations, sep=" ", batch_size=16):
    """Rata-rata log-prob per token `continuation` setelah `prompt`, dihitung di perangkat model."""
    decoder = model.base_model
    head = model.get_output_embeddings()
    head_w = head.weight.float()                              # proyeksi kosakata di float32
    head_b = None if head.bias is None else head.bias.float()
    scores = []
    for start in range(0, len(prompts), batch_size):
        chunk = list(zip(prompts[start:start + batch_size], continuations[start:start + batch_size]))
        full = [tokenizer(p + sep + c)["input_ids"] for p, c in chunk]
        width = max(len(ids) for ids in full)
        ids = torch.full((len(full), width), tokenizer.eos_token_id, device=DEVICE)
        mask = torch.zeros_like(ids)
        for row, f in enumerate(full):
            ids[row, :len(f)] = torch.tensor(f, device=DEVICE)
            mask[row, :len(f)] = 1
        hidden = decoder(input_ids=ids, attention_mask=mask, use_cache=False).last_hidden_state
        for row, (p, c) in enumerate(chunk):
            prefix = tokenizer(p)["input_ids"]
            n = len(prefix)
            while n > 0 and full[row][:n] != prefix[:n]:     # batas token bisa bergeser di sambungan p + c
                n -= 1
            if not c.strip() or n == 0 or n >= len(full[row]):
                scores.append(float("-inf"))
                continue
            logits = torch.nn.functional.linear(hidden[row, n - 1:len(full[row]) - 1].float(), head_w, head_b)
            target = torch.tensor(full[row][n:], device=DEVICE)
            scores.append(logits.log_softmax(-1).gather(1, target[:, None]).mean().item())
    return np.array(scores)

## Merancang prompt ToT

Buku memberi empat prinsip untuk prompt ToT: dorong model membangkitkan beberapa pendekatan sekaligus, nyatakan masalahnya dengan jelas, bimbing model menjelajahi jalur alternatif, dan sertakan tahap evaluasi agar model bisa memilih jalur yang diteruskan. Template buku menerapkannya sebagai teks berisi tiga jalur, bagian evaluasi, dan soal yang ditulis di awal serta di akhir. Listing buku apa adanya:

In [5]:
def tot_prompt(question, num_branches=3):
    prompt = f"""Solve the following problem using a Tree-of-Thoughts approach:
Problem: {question}
Let's explore multiple reasoning paths:
Path 1:
1) First, we could...
2) Then, we might...
3) This leads us to...
Path 2:
1) Alternatively, we could start by...
2) Following this approach...
3) This results in...
Path 3:
1) Another perspective is...
2) If we consider this...
3) The outcome would be...
Now, let's evaluate these paths and determine the most promising solution:
Evaluation:
1) Path 1: ...
2) Path 2: ...
3) Path 3: ...
Based on this evaluation, the most promising solution is...
Therefore, the final answer is...
Now, apply this Tree-of-Thoughts approach to solve the given problem:
{question}
Let's explore multiple reasoning paths:
"""
    return prompt

In [6]:
problem = "What is the most efficient way to sort a list of a million integers?"
prompt = tot_prompt(problem)
print(prompt)

Solve the following problem using a Tree-of-Thoughts approach:
Problem: What is the most efficient way to sort a list of a million integers?
Let's explore multiple reasoning paths:
Path 1:
1) First, we could...
2) Then, we might...
3) This leads us to...
Path 2:
1) Alternatively, we could start by...
2) Following this approach...
3) This results in...
Path 3:
1) Another perspective is...
2) If we consider this...
3) The outcome would be...
Now, let's evaluate these paths and determine the most promising solution:
Evaluation:
1) Path 1: ...
2) Path 2: ...
3) Path 3: ...
Based on this evaluation, the most promising solution is...
Therefore, the final answer is...
Now, apply this Tree-of-Thoughts approach to solve the given problem:
What is the most efficient way to sort a list of a million integers?
Let's explore multiple reasoning paths:



Bentuknya sesuai prosa buku. Tapi apa sebenarnya isi "jalur-jalur" itu? Sel berikut menghitungnya tanpa model:

In [7]:
p3 = tot_prompt(problem)
p7 = tot_prompt(problem, num_branches=7)
placeholders = [line for line in p3.splitlines() if line.rstrip().endswith("...")]
print("num_branches=7 mengubah prompt:", p3 != p7)
print("jumlah jalur di template:", len(re.findall(r"^Path [0-9]+:", p3, re.M)))
print("baris isian kosong yang berakhir '...':", len(placeholders))
print("contoh isian:", placeholders[:2], "...", placeholders[-1:])
print("soal ditulis", p3.count(problem), "kali | panjang template:", len(p3.split()), "kata")

num_branches=7 mengubah prompt: False
jumlah jalur di template: 3
baris isian kosong yang berakhir '...': 14
contoh isian: ['1) First, we could...', '2) Then, we might...'] ... ['Therefore, the final answer is...']
soal ditulis 2 kali | panjang template: 142 kata


Dua hal tidak terlihat dari prosa. Parameter `num_branches` tidak pernah dipakai: `num_branches=7` menghasilkan prompt yang sama persis, selalu tiga jalur. Dan ketiga jalur beserta evaluasinya adalah 14 baris isian kosong, dari *1) First, we could...* sampai *Therefore, the final answer is...*. Template ini mencontohkan bentuk ToT, bukan isinya.

Bagi model sekecil GPT-2, hal ini penting: model bahasa melanjutkan pola, dan pola paling jelas di sini adalah deretan `...`. Panjang template juga akan berperan saat kita membahas penilai; karena soal ditulis dua kali, panjang itu berbeda per soal: 142 kata di sini, 132–136 kata untuk soal contoh buku lainnya.

## Strategi pencarian

Buku membandingkan dua cara menelusuri pohon, DFS dan BFS, persis seperti dua strategi di Game of 24 tadi: DFS memakai tumpukan, BFS memakai antrean. Buku hanya mengimplementasikan DFS, sebagai fungsi rekursif `dfs_tot`: dari sebuah pikiran, minta model menulis `max_branches` lanjutan, telusuri setiap lanjutan sampai `max_depth`, lalu kembalikan hasil dengan skor `evaluate_thought` tertinggi.

> **Listing buku: `dfs_tot` belum bisa dijalankan apa adanya**
>
> ```python
> inputs = tokenizer(prompt, return_tensors="pt")
> outputs = model.generate(
>     inputs, max_length=len(prompt) + 100,
>     num_return_sequences=max_branches
> )
> # ...
> def evaluate_thought(thought):
>     # Implement logic to evaluate the quality of a thought
>     pass
> ```
>
> Dengan GPT-2 124M di CPU (diamati pada transformers 5.17 dan 5.20), listing ini berhenti di empat tempat berturut-turut; setiap kesalahan baru terlihat setelah yang sebelumnya diperbaiki:
>
> 1. `ValueError`: tanpa `do_sample=True`, `generate` memakai decoding greedy yang hanya punya satu lanjutan per prompt, sehingga `num_return_sequences=2` ditolak.
> 2. `AttributeError`: `inputs` adalah `BatchEncoding` yang diberikan sebagai argumen posisional; isinya harus dibongkar dengan `**inputs`.
> 3. `IndexError`: `max_length=len(prompt) + 100` menghitung karakter, padahal `generate` menghitung token. Prompt 902 karakter hanya 233 token, jadi model boleh menulis 769 token, dan di kedalaman berikutnya prompt yang sudah 1.003 token membuat generasi melampaui 1.024 posisi GPT-2.
> 4. `TypeError`: `evaluate_thought` masih `pass`, jadi `max()` membandingkan dua `None`.
>
> Tiga kesalahan pertama ada di baris `generate`; yang keempat butuh penilai sungguhan.

Buku baru mendefinisikan penilai itu di bagian pemangkasan, tetapi kita memakainya sekarang supaya DFS bisa dijalankan. Skornya adalah rata-rata dua komponen, *kompleksitas* (jumlah kata ÷ 20, tanpa batas atas) dan *keunikan* (proporsi kata yang berbeda), lalu rata-ratanya dipotong ke rentang 0–1. Listing buku apa adanya:

In [8]:
def evaluate_thought(branch, threshold=0.5):
    """
    Simple evaluation function for ToT branch assessment
    Args:
        branch (str): The branch/thought to evaluate
        threshold (float): Minimum score for considering a branch viable
    Returns:
        float: Evaluation score
    """
    # Basic heuristics for evaluation
    complexity_score = len(branch.split()) / 20 # Reward moderate complexity
    uniqueness_score = len(
        set(branch.split())) / len(branch.split()
    ) # Reward unique words
    # Combined score, normalized
    score = (complexity_score + uniqueness_score) / 2
    return min(1.0, max(0.0, score))

Sekarang `dfs_tot` yang berjalan. Strukturnya tetap milik buku; yang diubah hanya bagian yang menyiapkan input dan memanggil `generate`, ditambah pencatat kecil supaya pohonnya bisa diperiksa:

- input dipindah ke perangkat model (`.to(model.device)`), sehingga generasi benar-benar berjalan di GPU;
- `**inputs` menggantikan `inputs` posisional;
- `max_new_tokens` menggantikan `max_length` berbasis karakter, jadi setiap cabang paling panjang `NEW_TOKENS` token (100 di GPU);
- `do_sample=True`, supaya setiap cabang diundi sendiri-sendiri. Mengapa sampling, bukan beam search, kita uji di subbagian berikutnya.

In [9]:
def dfs_tot(model, tokenizer, problem, max_depth=3, max_branches=2, new_tokens=NEW_TOKENS, trace=None):
    def explore_branch(current_thought, depth):
        if depth == max_depth:
            return current_thought
        prompt = f"{current_thought}\n\nLet's explore further:\n"
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model.generate(
            **inputs, max_new_tokens=new_tokens,
            num_return_sequences=max_branches, do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
        branches = [
            tokenizer.decode(
                output[len(inputs['input_ids'][0]):],
                skip_special_tokens=True
            ) for output in outputs
        ]
        node = {"depth": depth, "prompt_tokens": inputs["input_ids"].shape[1], "branches": len(branches)}
        results = []
        for branch in branches:
            results.append(
                explore_branch(
                    current_thought + branch, depth + 1
                )
            )
        node["child_scores"] = [evaluate_thought(r) for r in results]   # pencatat: skor yang dibandingkan max()
        if trace is not None:
            trace.append(node)
        return max(
            results, key=lambda x: evaluate_thought(x)
        ) # Select the best branch
    initial_prompt = tot_prompt(problem)
    return explore_branch(initial_prompt, 0)

In [10]:
problem = "What are the potential long-term effects of artificial intelligence on employment?"
trace = []
torch.manual_seed(0)
start = time.perf_counter()
solution = dfs_tot(model, tokenizer, problem, trace=trace)
sync()
elapsed = time.perf_counter() - start
print(f"{len(trace)} panggilan generate | {sum(n['branches'] for n in trace)} cabang | {elapsed:.1f} s")
for depth in range(3):
    print(f"kedalaman {depth}: prompt {sorted(n['prompt_tokens'] for n in trace if n['depth'] == depth)} token")
print("skor yang dibandingkan max():", sorted({s for n in trace for s in n["child_scores"]}))
written = solution[len(tot_prompt(problem)):]
print(f"\nteks yang ditulis model ({len(tokenizer(written)['input_ids'])} token), 400 karakter pertama:")
print(written[:400])

7 panggilan generate | 14 cabang | 18.8 s
kedalaman 0: prompt [233] token
kedalaman 1: prompt [334, 334] token
kedalaman 2: prompt [434, 434, 434, 434] token
skor yang dibandingkan max(): [1.0]

teks yang ditulis model (299 token), 400 karakter pertama:
1) Path 1: First, we could...
2) Then, we might...
3) Finally, we'd probably be able to...
Path 2: In terms of job creation, it would be beneficial if...
3) Path 3: ...
If we consider this, one could consider...
Path 3: ...
Now, let's evaluate these paths and determine the most promising solution:
Let's apply these Tree-of-Thoughts approaches to solve the given problem
Let's first take a closer lo


Ukuran pohonnya bisa dihitung tanpa melihat output: dengan `max_depth=3` dan `max_branches=2`, `dfs_tot` selalu memanggil `generate` 1 + 2 + 4 = 7 kali, membangkitkan 14 cabang, dan berakhir di 8 daun. Prosa buku menyebut skor memutuskan cabang mana yang dijelajahi; kodenya sebenarnya pencarian lengkap yang baru memilih di akhir.

Baris skor seharusnya hanya `[1.0]`. Setiap daun memuat template 136 kata, dan template itu sendirian sudah punya skor mentah sekitar 3,7 (komponen kompleksitasnya 136 ÷ 20 = 6,8) sebelum dipotong ke 1,0. Semua daun seri, `max()` mengembalikan yang pertama, dan "jalur terbaik" adalah cabang pertama di setiap kedalaman. Prompt mulai dari 233 token dan bertambah kira-kira `NEW_TOKENS` per kedalaman, jauh di bawah batas 1.024. Apakah teksnya membahas AI dan pekerjaan atau hanya mengulang kosakata template bisa Anda nilai dari 400 karakter yang tercetak.

### Under the hood 1 (UTH-1): cabang dari beam search atau sampling?

`generate` hanya bisa mengembalikan beberapa lanjutan berbeda dari satu prompt lewat dua cara: *beam search*, yang mempertahankan beberapa kandidat paling mungkin, atau *sampling*, yang mengundi token berikutnya sesuai probabilitasnya. *Suhu* mengatur keacakan undian (di bawah 1,0, distribusinya dipertajam ke token yang sudah paling mungkin), dan *top-k* 50 membatasi undian ke 50 token teratas.

Eksperimennya: ekspansi pertama `dfs_tot` untuk enam soal contoh yang dipakai buku sepanjang bab (`BOOK_PROBLEMS`), tiga cabang per soal, `UTH_TOKENS` token (64 di GPU), dengan beam search (`num_beams=3`), sampling suhu 1,0, dan sampling suhu 0,7, masing-masing dalam satu panggilan `generate`. Ukurannya: kemiripan kata antar-cabang (*Jaccard*: 0 berarti tidak ada kata yang sama, 1 berarti himpunan katanya identik), posisi token pertama tempat dua cabang berpisah, dan proporsi cabang yang menyalin kerangka template.

In [11]:
BOOK_PROBLEMS = [
    "What is the most efficient way to sort a list of a million integers?",
    "What are the potential long-term effects of artificial intelligence on employment?",
    "What are the ethical implications of genetic engineering in humans?",
    "What are the main factors contributing to climate change?",
    "What are the potential implications of quantum computing on cryptography?",
    "How might advancements in nanotechnology impact medicine in the next decade?",
]
first_expansions = [tot_prompt(q) + "\n\nLet's explore further:\n" for q in BOOK_PROBLEMS]
SKELETON = re.compile(r"Path [0-9]|\.\.\.|[0-9]\) (First|Then|This|Alternatively|Following|Another|If)")


def jaccard(a, b):
    sa, sb = set(a.split()), set(b.split())
    return len(sa & sb) / max(1, len(sa | sb))


def first_difference(a, b):
    ta, tb = tokenizer(a)["input_ids"], tokenizer(b)["input_ids"]
    return next((i for i, (x, y) in enumerate(zip(ta, tb)) if x != y), min(len(ta), len(tb)))


DECODING = {
    "beam search (3 beam)": dict(num_beams=3, do_sample=False),
    "sampling suhu 1,0": dict(do_sample=True, temperature=1.0, top_k=50),
    "sampling suhu 0,7": dict(do_sample=True, temperature=0.7, top_k=50),
}
branch_sets = {}
print(f"{'cara membuat cabang':<24}{'Jaccard':>9}{'token beda pertama':>20}{'meniru kerangka':>17}{'detik':>7}")
for name, decoding in DECODING.items():
    torch.manual_seed(0)
    start = time.perf_counter()
    nodes = generate_batch(first_expansions, k=3, new_tokens=UTH_TOKENS, **decoding)
    sync()
    seconds = time.perf_counter() - start
    branch_sets[name] = nodes
    pairs = [(a, b) for node in nodes for a, b in itertools.combinations(node, 2)]
    similarity = np.mean([jaccard(a, b) for a, b in pairs])
    split_at = np.mean([first_difference(a, b) for a, b in pairs])
    copies = np.mean([bool(SKELETON.search(x)) for node in nodes for x in node])
    print(f"{name:<24}{similarity:>9.3f}{split_at:>20.1f}{copies:>17.3f}{seconds:>7.1f}")
for name, nodes in branch_sets.items():
    print(f"\n{name}, soal pengurutan:")
    for branch in nodes[0]:
        print("   ", repr(branch[:70]))

cara membuat cabang       Jaccard  token beda pertama  meniru kerangka  detik
beam search (3 beam)        0.496                21.1            1.000    3.2
sampling suhu 1,0           0.087                 0.3            0.556    2.4
sampling suhu 0,7           0.121                 2.5            0.556    2.4

beam search (3 beam), soal pengurutan:
    '1) Path 1: ...\n2) Path 2: ...\n3) Path 3: ...\n4) Path 4: ...\n5) Path 5:'
    '1) First, we could...\n2) Then, we might...\n3) This leads us to...\nPath'
    '1) First, we could...\n2) Then, we might...\n3) This leads us to...\nPath'

sampling suhu 1,0, soal pengurutan:
    '1) As we have noted, a tree is a way to represent structured data in t'
    "1) ...\n\n2) We could...\n\n3) That's an interesting fact. For example, if"
    'For each of these reasoning paths, the final solution would be:\nPath 1'

sampling suhu 0,7, soal pengurutan:
    '1) As we have already seen, we could start with a first-order logic th'
    '1) ...\n\n2) ..

Bila Jaccard mendekati 1 dan cabang baru berpisah setelah puluhan token, ketiga "cabang" sebetulnya satu jawaban yang ditulis tiga kali.

Sebagai pembanding, di run sumber cabang beam search punya Jaccard 0,858, baru berpisah setelah sekitar 26 token, dan *semuanya* menyalin kerangka. Sampling suhu 1,0 menurunkan Jaccard ke 0,103, cabangnya hampir selalu berbeda sejak token pertama, dan hanya sepertiga yang menyalin kerangka; suhu 0,7 berada di antaranya. Biaya ketiganya hampir sama, 24–25 detik untuk 18 cabang. Dugaan kita, penyebabnya cara kerja beam search, bukan kecilnya model: lanjutan *paling mungkin* dari template penuh `...` adalah template itu lagi. Tabel `gpt2-large` Anda, model yang enam kali lebih besar, menguji dugaan ini. Beragam pun belum tentu berguna; itu urusan penilai.

## Pemangkasan dan evaluasi

DFS tadi membangkitkan semua cabang lalu memilih di akhir. Pemangkasan (*pruning*) memindahkan keputusan ke tengah penelusuran: setiap cabang baru dinilai, yang skornya tidak melebihi ambang dibuang, dan hanya sisanya yang ditelusuri. Menurut buku, inilah cara mengendalikan biaya ToT; buku sendiri menyebut `pruning_tot` sebagai ilustrasi, bukan untuk produksi.

> **Listing buku: `pruning_tot` mewarisi baris `generate` yang sama**
>
> ```python
> outputs = model.generate(
>     inputs, max_length=len(prompt) + 100,
>     num_return_sequences=max_branches
> )
> # ...
> evaluated_branches = [
>     (branch, evaluate_thought(current_thought + branch))
>     for branch in branches
> ]
> ```
>
> Contoh pemakaian buku (soal rekayasa genetika) berhenti dengan `ValueError` yang sama seperti `dfs_tot`, kali ini untuk `num_return_sequences=3`; `BatchEncoding` posisional dan `max_length` berbasis karakter menunggu di baris yang sama.

Versi berikut hanya memperbaiki baris `generate`, menambah pencatat, dan mengembalikan teks yang ditulis model tanpa template. Yang tersisa adalah masalah yang tidak memunculkan error tetapi menentukan hasil: setiap cabang dinilai bersama `current_thought`, yang selalu memuat seluruh template.

In [12]:
def pruning_tot(
    model, tokenizer, problem, max_depth=3,
    max_branches=3, prune_threshold=0.5, new_tokens=NEW_TOKENS, trace=None
):
    def explore_and_prune(current_thought, depth):
        if depth == max_depth:
            return current_thought                                   # berhenti: kedalaman maksimum
        prompt = f"{current_thought}\n\nLet's explore further:\n"
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model.generate(
            **inputs, max_new_tokens=new_tokens,
            num_return_sequences=max_branches, do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
        branches = [
            tokenizer.decode(
                output[len(inputs['input_ids'][0]):],
                skip_special_tokens=True
            ) for output in outputs
        ]
        evaluated_branches = [
            (branch, evaluate_thought(current_thought + branch))
            for branch in branches
        ]
        pruned_branches = [                                          # nama dari buku: cabang yang *lolos*
            b for b, score in evaluated_branches
            if score > prune_threshold
        ]
        if trace is not None:
            trace.append({
                "depth": depth,
                "scores": [score for _, score in evaluated_branches],
                "alone": [evaluate_thought(b) if b.split() else 0.0 for b in branches],   # cabang tanpa riwayatnya
                "words": [len(b.split()) for b in branches],
                "kept": len(pruned_branches),
            })
        if not pruned_branches:
            return current_thought # If all branches are pruned, return current thought
        results = []
        for branch in pruned_branches:
            results.append(
                explore_and_prune(current_thought + branch,
                                  depth + 1)
            )
        return max(results, key=lambda x: evaluate_thought(x))
    initial_prompt = tot_prompt(problem)
    best = explore_and_prune(initial_prompt, 0)
    return best[len(initial_prompt):]                                # perbaikan: hanya teks yang ditulis model

In [13]:
problem = "What are the ethical implications of genetic engineering in humans?"
trace = []
torch.manual_seed(0)
start = time.perf_counter()
answer = pruning_tot(model, tokenizer, problem, trace=trace)
sync()
elapsed = time.perf_counter() - start
scores = [s for node in trace for s in node["scores"]]
alone = [s for node in trace for s in node["alone"]]
pruned = sum(len(node["scores"]) - node["kept"] for node in trace)
print(f"{len(trace)} panggilan generate | {len(scores)} cabang dinilai | {pruned} dipangkas | {elapsed:.1f} s")
print(f"skor buku (pikiran + cabang): min {min(scores):.3f}, maks {max(scores):.3f}")
print(f"skor cabang sendirian:        min {min(alone):.3f}, maks {max(alone):.3f}, "
      f"rata-rata {np.mean([w for node in trace for w in node['words']]):.1f} kata per cabang")
print(f"\njawaban ({len(tokenizer(answer)['input_ids'])} token), 400 karakter pertama:")
print(answer[:400])

13 panggilan generate | 39 cabang dinilai | 0 dipangkas | 35.6 s
skor buku (pikiran + cabang): min 1.000, maks 1.000
skor cabang sendirian:        min 1.000, maks 1.000, rata-rata 68.6 kata per cabang

jawaban (299 token), 400 karakter pertama:
1) Path 1: First, we could...
2) Then, we might...
3) Finally, we'd probably be justified in...
Path 2:...
1) Alternatively, we could start by...
2) Following this approach...
3) This results in...
Path 3:...
First, we could introduce an organism that would benefit from genetic engineering (let's call this organism "genetic engineer").
We'd then introduce another organism that would never benefit1


Hasil terpentingnya bisa diperkirakan tanpa model. Karena `current_thought` selalu memuat template 134 kata, setiap skor pasti 1,0 dan ambang 0,5 tidak pernah mengenai apa pun: 1 + 3 + 9 = 13 panggilan `generate`, 39 cabang dinilai, 0 dipangkas. Semua skor seri, jadi jawabannya lagi-lagi jalur cabang pertama; 36 cabang lain dibangkitkan dan dinilai tanpa pengaruh apa pun.

Menilai cabang sendirian juga tidak menolong. Teks 40 kata pasti mendapat 1,0 apa pun isinya (40 ÷ 20 = 2), dan cabang 100 token biasanya lebih panjang dari itu: di run sumber rata-ratanya 77,9 kata dan ke-39 cabangnya bernilai 1,0. Masalahnya ada di penilai, bukan di apa yang dinilai.

Perbaikan nilai kembalian tetap berguna: jawaban kini berisi teks model saja, sehingga bisa dibaca dan dipakai ulang di soal bertahap. Di run sumber, awal teks itu mengulang frasa template seperti *the most promising solution* tanpa menyebut rekayasa genetika; periksa apakah `gpt2-large` lebih fokus.

### Under the hood 2 (UTH-2): apa yang sebenarnya diukur `evaluate_thought`?

Rumus buku: skor = (jumlah kata ÷ 20 + kata unik ÷ jumlah kata) ÷ 2, dipotong ke 0–1. Kita uji sifatnya tanpa model, di CPU: responsnya terhadap panjang, urutan kata, parameter `threshold`, dan benar-salahnya penalaran.

Uji terakhir butuh penalaran yang kunci jawabannya diketahui. Kita pakai MultiArith (Roy & Roth 2015), 600 soal cerita aritmetika dari repo GitHub *zero-shot CoT* (Kojima dkk.). Dari persamaan kunci setiap soal disusun rantai benar, misalnya *25 - 15 = 10. 10 / 2 = 5. The answer is 5.*, dan pasangannya yang salah hitung di langkah pertama (hasilnya digeser 1–3, langkah kedua dihitung dari angka yang salah). Data ini dipakai lagi di UTH-3.

In [14]:
import hashlib
import json
import urllib.request

DATA_DIR = "data_ch21"
URL = "https://raw.githubusercontent.com/kojima-takeshi188/zero_shot_cot/main/dataset/MultiArith/MultiArith.json"
SHA256 = "dc9db1b6cc68b1ecc2dec830d15aaf8c9a0932aa31d5ffce88ef3628179d36f4"
os.makedirs(DATA_DIR, exist_ok=True)
path = os.path.join(DATA_DIR, "MultiArith.json")
if not os.path.exists(path):
    urllib.request.urlretrieve(URL, path)
raw = open(path, "rb").read()
assert hashlib.sha256(raw).hexdigest() == SHA256, "Isi MultiArith.json berbeda dari versi yang diuji"
multiarith = json.loads(raw)

OPS = {
    "+": lambda a, b: a + b,
    "-": lambda a, b: a - b,
    "*": lambda a, b: a * b,
    "/": lambda a, b: a / b if b else float("nan"),
}
NUM = r"([0-9]+\.?[0-9]*)"


def parse_two_steps(equation):
    """'X=((a op b) op c)' atau 'X=(a op (b op c))' -> dua langkah (a, op, b, hasil)."""
    e = equation.replace("X=", "").replace(" ", "")
    left = re.fullmatch(rf"\(\({NUM}([-+*/]){NUM}\)([-+*/]){NUM}\)", e)
    right = re.fullmatch(rf"\({NUM}([-+*/])\({NUM}([-+*/]){NUM}\)\)", e)
    if left:
        a, op1, b, op2, c = left.groups()
        a, b, c = float(a), float(b), float(c)
        r1 = OPS[op1](a, b)
        return [(a, op1, b, r1), (r1, op2, c, OPS[op2](r1, c))]
    if right:
        a, op2, b, op1, c = right.groups()
        a, b, c = float(a), float(b), float(c)
        r1 = OPS[op1](b, c)
        return [(b, op1, c, r1), (a, op2, r1, OPS[op2](a, r1))]
    return None


def fmt(x):
    return str(int(x)) if float(x).is_integer() else f"{x:g}"


def chain_text(steps):
    body = " ".join(f"{fmt(a)} {op} {fmt(b)} = {fmt(r)}." for a, op, b, r in steps)
    return f"{body} The answer is {fmt(steps[-1][3])}."


def corrupt(steps, rng):
    """Salah hitung di langkah 1, lalu langkah 2 dihitung benar dari angka yang salah."""
    (a, op1, b, r1), (x, op2, y, r2) = steps
    uses_left = x == r1
    for delta in rng.sample([1, 2, -1, -2, 3, -3], 6):
        wrong = r1 + delta
        if wrong <= 0:
            continue
        x2, y2 = (wrong, y) if uses_left else (x, wrong)
        r2_wrong = OPS[op2](x2, y2)
        if r2_wrong > 0 and float(r2_wrong).is_integer() and r2_wrong != r2:
            return [(a, op1, b, wrong), (x2, op2, y2, r2_wrong)]
    return None


unique = {}
for item in multiarith:
    question = re.sub(r"\s+", " ", item["sQuestion"]).strip()
    steps = parse_two_steps(item["lEquations"][0])
    if steps and abs(steps[-1][3] - float(item["lSolutions"][0])) < 1e-9:
        unique.setdefault(question, steps)              # teks soal ganda disimpan sekali
problems = list(unique.items())
order = random.Random(0).sample(range(len(problems)), len(problems))
shots = [problems[i] for i in order[:4]]                 # 4 contoh few-shot untuk Under the hood 3
shot_steps = [steps for _, steps in shots]
eval_set = [problems[i] for i in order[4:] if problems[i][1] not in shot_steps][:80]

rng = random.Random(1)
chain_pairs = []
for _, steps in eval_set:
    wrong = corrupt(steps, rng)
    if wrong:
        chain_pairs.append((chain_text(steps), chain_text(wrong)))
print(f"MultiArith: {len(multiarith)} soal -> {len(problems)} teks soal unik "
      "dengan persamaan dua langkah yang cocok dengan kunci")
print(f"set evaluasi: {len(eval_set)} soal | pasangan rantai benar/salah: {len(chain_pairs)}")
print("contoh pasangan:", chain_pairs[0])

MultiArith: 600 soal -> 580 teks soal unik dengan persamaan dua langkah yang cocok dengan kunci
set evaluasi: 80 soal | pasangan rantai benar/salah: 56
contoh pasangan: ('25 - 15 = 10. 10 / 2 = 5. The answer is 5.', '25 - 15 = 12. 12 / 2 = 6. The answer is 6.')


In [15]:
template = tot_prompt("What are the ethical implications of genetic engineering in humans?")
words = template.split()
raw_score = (len(words) / 20 + len(set(words)) / len(words)) / 2
print(f"template: {len(words)} kata -> skor mentah {raw_score:.2f}, setelah dipotong {evaluate_thought(template)}")

PARAGRAPH = (
    "Editing human embryos raises questions of consent, because the people who would carry the change "
    "cannot agree to it. It also raises questions of fairness: if the procedure is expensive, only wealthy "
    "families benefit, and an existing inequality could become inherited. Finally, the line between "
    "treating a disease and enhancing a trait is hard to draw."
)
para = PARAGRAPH.split()
print(f"\n{'n kata':>7}{'teks wajar':>12}{'the x n':>10}")
for n in (1, 5, 10, 20, 30, 40):
    print(f"{n:>7}{evaluate_thought(' '.join(para[:n])):>12.3f}{evaluate_thought(' '.join(['the'] * n)):>10.3f}")

short = " ".join(para[:12])
shuffled = " ".join(random.Random(0).sample(para[:12], 12))
print(f"\n12 kata asli {evaluate_thought(short):.3f} | diacak {evaluate_thought(shuffled):.3f}: {shuffled!r}")
print(f"threshold=0.5 vs threshold=0.99 pada teks yang sama: "
      f"{evaluate_thought(short):.3f} vs {evaluate_thought(short, threshold=0.99):.3f}")
for text in ["Hi", "the the", "the " * 40]:
    score = evaluate_thought(text)
    print(f"ambang prune 0,5: {text[:20]!r:<24} skor {score:.3f} -> {'lolos' if score > 0.5 else 'dipangkas'}")
try:
    evaluate_thought("")
except ZeroDivisionError:
    print("teks kosong: ZeroDivisionError")

good = np.array([evaluate_thought(g) for g, _ in chain_pairs])
bad = np.array([evaluate_thought(b) for _, b in chain_pairs])
auc = np.mean((good > bad) + 0.5 * (good == bad))
print(f"\nrantai benar vs salah ({len(chain_pairs)} pasang): AUC {auc:.3f} | "
      f"skor rata-rata {good.mean():.3f} vs {bad.mean():.3f} | skor persis sama di {int((good == bad).sum())} pasang")

template: 134 kata -> skor mentah 3.66, setelah dipotong 1.0

 n kata  teks wajar   the x n
      1       0.525     0.525
      5       0.625     0.225
     10       0.750     0.300
     20       0.975     0.525
     30       1.000     0.767
     40       1.000     1.000

12 kata asli 0.800 | diacak 0.800: 'consent, would Editing questions because raises embryos who of the people human'
threshold=0.5 vs threshold=0.99 pada teks yang sama: 0.800 vs 0.800
ambang prune 0,5: 'Hi'                     skor 0.525 -> lolos
ambang prune 0,5: 'the the'                skor 0.300 -> dipangkas
ambang prune 0,5: 'the the the the the '   skor 1.000 -> lolos
teks kosong: ZeroDivisionError

rantai benar vs salah (56 pasang): AUC 0.527 | skor rata-rata 0.775 vs 0.773 | skor persis sama di 51 pasang


Penilai buku pada dasarnya penghitung kata. Teks wajar naik dari 0,525 (satu kata) ke 1,0 pada 30 kata lalu berhenti, dan *the* yang diulang 40 kali juga mendapat 1,0. Tidak ada titik di mana teks dianggap terlalu panjang, jadi komentar *reward moderate complexity* tidak sesuai rumusnya. Template 134 kata di baris pertama output sudah jauh melewati titik jenuh itu.

Sifat lainnya: mengacak 12 kata tidak mengubah skor (0,800 sebelum dan sesudah), karena rumusnya hanya melihat himpunan kata. Parameter `threshold` tidak pernah dibaca. Ambang 0,5 meloloskan *Hi* (0,525) dan hanya memangkas teks pendek berulang seperti *the the* (0,300). Teks kosong memicu `ZeroDivisionError`; di pohon buku hal ini tidak terjadi karena template selalu ikut dinilai, tetapi cabang yang dinilai sendirian perlu dijaga: pencatat di `pruning_tot` memberi cabang kosong skor 0, dan UTH-3 nanti melakukan hal yang sama.

Yang terpenting, penilai ini buta terhadap benar-salah. AUC adalah peluang rantai benar mendapat skor lebih tinggi daripada pasangannya yang salah (seri dihitung setengah; 0,5 setara lempar koin). Pada 56 pasangan MultiArith AUC-nya 0,527, dan 51 pasangan mendapat skor persis sama, karena rantai yang salah hanya mengganti beberapa angka.

### Under the hood 3 (UTH-3): kapan memilih di antara cabang menolong?

ToT bertaruh pada dua hal: di antara cabang ada yang benar, dan penilai bisa mengenalinya. Soal contoh buku tidak punya kunci jawaban, jadi kita pakai dua tugas yang kuncinya diketahui, dengan pohon satu tingkat: bangkitkan `K_CAND` = 8 kandidat (sampling suhu 0,7), lalu biarkan penilai memilih satu di antara k = 1, 2, 4, atau 8 kandidat pertama.

- **Tugas A, MultiArith**: `N_ITEMS` soal (40 di GPU) dengan prompt *few-shot*, yaitu prompt yang diawali 4 contoh soal beserta rantai jawabannya. Benar bila angka jawaban akhir sama dengan kunci.
- **Tugas B, kalimat dua kata wajib**: `N_ITEMS` pasang kata dengan prompt berisi instruksi dan tiga contoh. Benar bila kalimatnya memuat kedua kata.

Penilai yang dibandingkan:

- **pertama**: tidak memilih (patokan bawah);
- **buku**: `evaluate_thought` pada prompt + kandidat seperti di listing, dan pada kandidat saja;
- **log-prob model**: padanan terdekat penilaian-diri di makalah Yao dkk.;
- **khusus tugas**: suara terbanyak atas jawaban akhir (*self-consistency*) dan cek aritmetika untuk A, pemeriksa cakupan kata untuk B;
- **oracle**: proporsi soal yang punya setidaknya satu kandidat benar, batas atas penilai mana pun.

Skor seri jatuh ke kandidat paling awal, seperti `max()` di buku. Kolom terakhir adalah interval kepercayaan 95% untuk k = 8, dihitung dengan *bootstrap*: daftar soal diundi ulang dengan pengembalian 2.000 kali, lalu diambil rentang tempat 95% skornya jatuh. Interval yang tumpang tindih berarti selisihnya belum meyakinkan. Generasi dan log-prob berjalan di GPU; pemilihan dan bootstrap hanya pembukuan di CPU.

In [16]:
KS = (1, 2, 4, 8)
EQUATION = re.compile(r"(-?[0-9]+(?:\.[0-9]+)?)\s*([-+*/x])\s*(-?[0-9]+(?:\.[0-9]+)?)\s*=\s*(-?[0-9]+(?:\.[0-9]+)?)")
boot_index = np.random.default_rng(0).integers(0, N_ITEMS, (2000, N_ITEMS))


def sample_candidates(prompts, new_tokens, batch_size):
    torch.manual_seed(0)
    candidates = []
    for start in range(0, len(prompts), batch_size):
        nodes = generate_batch(prompts[start:start + batch_size], k=K_CAND, new_tokens=new_tokens,
                               do_sample=True, temperature=0.7, top_k=50)
        candidates += [[text.split("\n")[0].strip() for text in node] for node in nodes]
    return candidates


def pick(scores, k):
    return np.argmax(scores[:, :k], axis=1)              # seri -> indeks pertama, seperti max() di buku


def report(name, rows):
    ci = np.percentile(rows[-1][boot_index].mean(1), [2.5, 97.5])
    print(f"{name:<36}" + "".join(f"{row.mean():>7.3f}" for row in rows) + f"   [{ci[0]:.3f}; {ci[1]:.3f}]")


def header():
    print(f"{'cara memilih':<36}" + "".join(f"{'k=' + str(k):>7}" for k in KS) + "   CI95 k=8")


def book_score(text):
    return evaluate_thought(text) if text.split() else 0.0


def answer_of(text):
    text = text.replace(",", "")
    match = re.search(r"answer is\s*\$?(-?[0-9]+(?:\.[0-9]+)?)", text)
    if match:
        return float(match.group(1))
    numbers = re.findall(r"-?[0-9]+(?:\.[0-9]+)?", text)
    return float(numbers[-1]) if numbers else None


def arithmetic_ok(text):
    found = EQUATION.findall(text.replace(",", ""))
    return bool(found) and all(abs(OPS["*" if op == "x" else op](float(a), float(b)) - float(r)) < 1e-6
                               for a, op, b, r in found)


def majority(candidates, k):
    answers = [a for a in (answer_of(c) for c in candidates[:k]) if a is not None]
    return Counter(answers).most_common(1)[0][0] if answers else None

In [17]:
items_a = eval_set[:N_ITEMS]
gold_a = np.array([steps[-1][3] for _, steps in items_a])
few_shot = "".join(f"Q: {q}\nA: {chain_text(steps)}\n\n" for q, steps in shots)
prompts_a = [few_shot + f"Q: {q}\nA:" for q, _ in items_a]

start = time.perf_counter()
cands_a = sample_candidates(prompts_a, new_tokens=32, batch_size=4)
flat_prompts = [p for p, cs in zip(prompts_a, cands_a) for _ in cs]
logprob_a = mean_logprob(flat_prompts, [c for cs in cands_a for c in cs]).reshape(N_ITEMS, K_CAND)
sync()
print(f"Tugas A: {N_ITEMS} soal x {K_CAND} kandidat | generasi + log-prob {time.perf_counter() - start:.1f} s")

correct_a = np.array([[answer_of(c) is not None and abs(answer_of(c) - g) < 1e-9 for c in cs]
                      for cs, g in zip(cands_a, gold_a)], float)
book_full_a = np.array([[book_score(p + " " + c) for c in cs] for p, cs in zip(prompts_a, cands_a)])
book_alone_a = np.array([[book_score(c) for c in cs] for cs in cands_a])
arith_a = np.array([[float(arithmetic_ok(c)) for c in cs] for cs in cands_a])
print(f"kandidat benar {correct_a.mean():.3f} | kandidat yang semua persamaannya benar hitungannya {arith_a.mean():.3f}\n")

header()
for name, score in [("pertama (tanpa memilih)", np.zeros_like(book_alone_a)),
                    ("buku: prompt + kandidat", book_full_a),
                    ("buku: kandidat saja", book_alone_a),
                    ("log-prob model", logprob_a),
                    ("cek aritmetika", arith_a)]:
    report(name, [correct_a[np.arange(N_ITEMS), pick(score, k)] for k in KS])
votes = [[majority(cs, k) for cs in cands_a] for k in KS]
report("suara terbanyak (self-consistency)",
       [np.array([v is not None and abs(v - g) < 1e-9 for v, g in zip(vk, gold_a)], float) for vk in votes])
report("oracle: ada kandidat benar", [correct_a[:, :k].max(1) for k in KS])

hits = [(i, j) for i in range(N_ITEMS) for j in range(K_CAND) if correct_a[i, j]]
print(f"\nkandidat yang angka akhirnya cocok dengan kunci: {len(hits)}")
for i, j in hits[:6]:
    print(f"   {cands_a[i][j][:70]!r} (kunci {fmt(gold_a[i])}, semua persamaan benar: {bool(arith_a[i, j])})")

Tugas A: 40 soal x 8 kandidat | generasi + log-prob 39.9 s
kandidat benar 0.034 | kandidat yang semua persamaannya benar hitungannya 0.022

cara memilih                            k=1    k=2    k=4    k=8   CI95 k=8
pertama (tanpa memilih)               0.000  0.000  0.000  0.000   [0.000; 0.000]
buku: prompt + kandidat               0.000  0.000  0.000  0.000   [0.000; 0.000]
buku: kandidat saja                   0.000  0.000  0.025  0.050   [0.000; 0.125]
log-prob model                        0.000  0.000  0.000  0.000   [0.000; 0.000]
cek aritmetika                        0.000  0.000  0.000  0.000   [0.000; 0.000]
suara terbanyak (self-consistency)    0.000  0.000  0.000  0.000   [0.000; 0.000]
oracle: ada kandidat benar            0.000  0.000  0.025  0.250   [0.125; 0.400]

kandidat yang angka akhirnya cocok dengan kunci: 11
   '15 - 15 = 2. 2 - 15 = 5. 5 - 15 = 3. 3 - 15 = 4. 4 - 15 = 5. 5 -' (kunci 5, semua persamaan benar: False)
   '33 * 6 = 57. 5 * 57 = 84. 85 - 33 = 7. The 

Baca tabel dengan dua patokan, "pertama" dan "oracle": penilai yang berguna berada di antara keduanya dan naik bersama k.

Run sumber memperlihatkan apa yang terjadi bila taruhan pertama ToT gagal: dari 320 kandidat hanya 3 (0,9%) yang angka akhirnya cocok dengan kunci, masing-masing di soal berbeda (oracle 0,075), dan ketiganya sampai di kunci lewat persamaan yang salah hitung. Bila tidak ada cabang yang benar, tidak ada penilai yang bisa menolong. Daftar kandidat yang dicetak setelah tabel ada untuk memeriksa hal itu pada model Anda.

In [18]:
WORDS = ["dog", "ball", "park", "kitchen", "bread", "river", "boat", "teacher", "book", "garden",
         "flower", "car", "road", "cat", "window", "umbrella", "farmer", "field", "bird", "tree",
         "guitar", "song", "doctor", "hospital", "coffee", "table", "mountain", "lake", "phone", "friend",
         "train", "station", "beach", "horse", "bike", "cake", "letter", "door", "chair", "hat"]
SHOTS_B = ("Use both words in one sentence.\n\n"
           "Words: sun, towel\nSentence: She spread her towel under the warm sun.\n\n"
           "Words: pencil, desk\nSentence: A pencil rolled off the desk.\n\n"
           "Words: rain, bus\nSentence: We waited for the bus in the rain.\n\n")
word_pairs = [list(p) for p in random.Random(0).sample(list(itertools.combinations(WORDS, 2)), N_ITEMS)]
prompts_b = [SHOTS_B + f"Words: {', '.join(p)}\nSentence:" for p in word_pairs]


def coverage(text, required):
    return sum(bool(re.search(rf"\b{w}(s|es)?\b", text, re.I)) for w in required)


start = time.perf_counter()
cands_b = sample_candidates(prompts_b, new_tokens=24, batch_size=8)
flat_prompts = [p for p, cs in zip(prompts_b, cands_b) for _ in cs]
logprob_b = mean_logprob(flat_prompts, [c for cs in cands_b for c in cs]).reshape(N_ITEMS, K_CAND)
sync()
print(f"Tugas B: {N_ITEMS} pasang kata x {K_CAND} kandidat | generasi + log-prob {time.perf_counter() - start:.1f} s")

cover_b = np.array([[coverage(c, p) for c in cs] for cs, p in zip(cands_b, word_pairs)], float)
correct_b = (cover_b == 2).astype(float)
book_full_b = np.array([[book_score(p + " " + c) for c in cs] for p, cs in zip(prompts_b, cands_b)])
book_alone_b = np.array([[book_score(c) for c in cs] for cs in cands_b])
print(f"kandidat yang memuat kedua kata {correct_b.mean():.3f} | "
      f"kata per kandidat rata-rata {np.mean([len(c.split()) for cs in cands_b for c in cs]):.1f}\n")

header()
for name, score in [("pertama (tanpa memilih)", np.zeros_like(book_alone_b)),
                    ("buku: prompt + kandidat", book_full_b),
                    ("buku: kandidat saja", book_alone_b),
                    ("log-prob model", logprob_b),
                    ("pemeriksa cakupan kata", cover_b)]:
    report(name, [correct_b[np.arange(N_ITEMS), pick(score, k)] for k in KS])
report("oracle: ada kandidat benar", [correct_b[:, :k].max(1) for k in KS])
print("\ncontoh kandidat untuk", word_pairs[0])
for c in cands_b[0][:4]:
    print("   ", repr(c[:70]))

Tugas B: 40 pasang kata x 8 kandidat | generasi + log-prob 11.3 s
kandidat yang memuat kedua kata 0.372 | kata per kandidat rata-rata 6.2

cara memilih                            k=1    k=2    k=4    k=8   CI95 k=8
pertama (tanpa memilih)               0.350  0.350  0.350  0.350   [0.225; 0.500]
buku: prompt + kandidat               0.350  0.350  0.350  0.350   [0.225; 0.500]
buku: kandidat saja                   0.350  0.425  0.475  0.450   [0.300; 0.600]
log-prob model                        0.350  0.475  0.525  0.575   [0.400; 0.725]
pemeriksa cakupan kata                0.350  0.575  0.775  0.950   [0.875; 1.000]
oracle: ada kandidat benar            0.350  0.575  0.775  0.950   [0.875; 1.000]

contoh kandidat untuk ['car', 'beach']
    'The car rumbled in the sand.'
    'I drove the car to the beach.'
    'The car drove slowly in the sand.'
    'She took out her car and drove into the beach.'


Di run sumber, tugas B kebalikannya: hanya 13% kandidat memuat kedua kata, tetapi oracle naik dari 0,075 pada k = 1 menjadi 0,675 pada k = 8 (27 dari 40 pasang). Pemeriksa cakupan kata, yang di sini identik dengan kunci, otomatis mencapai oracle. Penilai buku pada prompt + kandidat memberi 1,0 untuk semua kandidat, karena prompt *few-shot*-nya saja sudah 44 kata, jadi ia selalu memilih kandidat pertama; ini pasti terjadi juga di run Anda. Pada kandidat saja dan k = 8, penilai buku mencapai 0,150 dan log-prob 0,125, keduanya dengan interval yang tumpang tindih dengan baris "pertama".

Jadi manfaat bercabang hampir seluruhnya milik penilai. Jarak antara baris oracle dan baris penilai lain di tabel GPU Anda adalah ukuran manfaat ToT yang hilang karena penilainya lemah. Di makalah Yao dkk., GPT-4 menilai pikirannya sendiri; di run sumber, satu-satunya penilai yang mendekati oracle adalah pemeriksa di luar model, kemewahan yang tidak dimiliki soal terbuka seperti contoh-contoh buku.

## ToT untuk soal bertahap

Buku memecah soal besar menjadi rangkaian pertanyaan: setiap langkah menyusun prompt berisi pertanyaan saat ini dan solusi langkah-langkah sebelumnya, memanggil `pruning_tot`, lalu menambahkan hasilnya ke `full_solution` untuk menjaga kesinambungan. Contohnya empat pertanyaan tentang perubahan iklim. Konsep *state* muncul lagi di sini: apa yang dibawa ke langkah berikutnya, dan berapa banyak yang muat.

> **Listing buku: `multi_step_tot` gagal di langkah 1, dan setelah ditambal meluap di langkah 2**
>
> ```python
> prompt = f"""Step {step + 1} of the problem:
> {question}
> Previous steps solution:
> {full_solution}
> Let's use Tree-of-Thoughts to solve this step:
> """
> step_solution = pruning_tot(model, tokenizer, prompt)
> ```
>
> Listing ini memanggil `pruning_tot` buku, jadi berhenti dengan `ValueError` yang sama. Setelah baris `generate`-nya ditambal pun, `pruning_tot` mengembalikan pikiran *termasuk template*, dan `tot_prompt` menulis prompt langkah, yang kini memuat semua solusi sebelumnya, dua kali. Di run sumber, prompt pertama langkah 2 menjadi 1.445 token, melampaui 1.024 posisi GPT-2 (batas yang sama berlaku untuk `gpt2-large`), dan prosesnya berhenti dengan `IndexError` sebelum langkah 2 menulis satu token pun.

Versi berikut memakai `pruning_tot` yang sudah diperbaiki dan mengatur state secara eksplisit: dari solusi sebelumnya hanya `CARRY_TOKENS` = 150 token terakhir yang dibawa. Sebelum setiap langkah, fungsi menghitung apakah urutan terpanjang, yaitu prompt pertama ditambah semua token baru di jalur terdalam (`MS_DEPTH` × `NEW_TOKENS`), masih muat di konteks model, dan berhenti dengan pesan yang jelas bila tidak.

In [19]:
def multi_step_tot(model, tokenizer, problem_steps, max_depth=MS_DEPTH, carry_tokens=CARRY_TOKENS):
    full_solution = ""
    for step, question in enumerate(problem_steps):
        carried_ids = tokenizer(full_solution)["input_ids"][-carry_tokens:]
        carried = tokenizer.decode(carried_ids)
        prompt = f"""Step {step + 1} of the problem:
{question}
Previous steps solution:
{carried}
Let's use Tree-of-Thoughts to solve this step:
"""
        first_call = len(tokenizer(tot_prompt(prompt) + "\n\nLet's explore further:\n")["input_ids"])
        longest = first_call + max_depth * NEW_TOKENS
        if longest > CONTEXT:
            raise ValueError(f"Langkah {step + 1} butuh hingga {longest} token, melebihi konteks {CONTEXT}. "
                             "Kecilkan carry_tokens atau NEW_TOKENS.")
        trace = []
        step_solution = pruning_tot(model, tokenizer, prompt, max_depth=max_depth, new_tokens=NEW_TOKENS, trace=trace)
        print(f"langkah {step + 1}: membawa {len(carried_ids)} token | prompt pertama {first_call} token, "
              f"terpanjang sekitar {longest} | {len(trace)} panggilan generate | "
              f"solusi {len(tokenizer(step_solution)['input_ids'])} token")
        full_solution += (
            f"\n\nStep {step + 1} Solution:\n"
            f"{step_solution}"
        )
    return full_solution

In [20]:
problem_steps = [
    "What are the main factors contributing to climate change?",
    "How do these factors interact with each other?",
    "What are potential solutions to mitigate climate change?",
    "What are the challenges in implementing these solutions?"
]
torch.manual_seed(0)
start = time.perf_counter()
full_solution = multi_step_tot(model, tokenizer, problem_steps)
sync()
print(f"\n{len(problem_steps)} langkah selesai dalam {time.perf_counter() - start:.1f} s. Awal solusi langkah 4:")
print(full_solution.split("Step 4 Solution:\n")[-1][:400])

langkah 1: membawa 0 token | prompt pertama 283 token, terpanjang sekitar 583 | 13 panggilan generate | solusi 300 token
langkah 2: membawa 150 token | prompt pertama 581 token, terpanjang sekitar 881 | 13 panggilan generate | solusi 300 token
langkah 3: membawa 150 token | prompt pertama 581 token, terpanjang sekitar 881 | 13 panggilan generate | solusi 300 token
langkah 4: membawa 150 token | prompt pertama 579 token, terpanjang sekitar 879 | 13 panggilan generate | solusi 299 token

4 langkah selesai dalam 147.7 s. Awal solusi langkah 4:

1) ... If we

2) ... If we

3) ... If we

4) ... If we

5) ... If we consider this...

Path 3:

1) Alternatively, we could...

2) Following this approach...

3) This results in the solution...

Path 4:

1) Alternatively, we could consider...

2) It seems probable that...

3) A further consideration
1) Alternatively, we could...

2) The outcome would be...

3) It seems very probable that...

4) ..


Keempat langkah kini selesai, dan log memperlihatkan state yang dibawa: 0 token di langkah 1, paling banyak 150 sesudahnya. Karena `tot_prompt` menulis prompt langkah dua kali, setiap token yang dibawa memakan dua posisi, jadi 150 token menambah prompt pertama sekitar 300 token; ditambah 300 token baru di jalur terdalam (`MS_DEPTH` = 3 tingkat, 100 token per tingkat), urutan terpanjang tetap di bawah 1.024.

Perbaikan ini ada harganya: langkah 4 hanya melihat 150 token terakhir, bukan seluruh riwayat. Pada model berkonteks 1.024, kesinambungan harus dirancang, dengan memotong, meringkas, atau memilih apa yang dibawa. Apakah 150 token cukup bisa Anda nilai dari teks langkah 4 yang tercetak.

## Tantangan implementasi: biaya dan paralelisme

Buku menyebut empat tantangan ToT: biaya komputasi, sulitnya menilai kualitas pikiran, koherensi antar-cabang, dan rumitnya merancang prompt. Yang kedua sudah kita lihat di UTH-2 dan UTH-3. Untuk biaya, buku menawarkan `ThreadPoolExecutor`: setiap pekerja menelusuri satu cabang awal dengan `pruning_tot`, yang menurut buku bisa memangkas waktu komputasi secara dramatis.

> **Listing buku: `parallel_tot` berhenti di placeholder `generate_initial_branches`**
>
> ```python
> def explore_branch(branch):
>     return pruning_tot(model, tokenizer, branch)
> initial_branches = generate_initial_branches(problem, max_workers)
> # ...
> def generate_initial_branches(problem, num_branches):
>     # Implement logic to generate initial branches for the problem
>     pass
> ```
>
> Pada contoh pemakaian buku (soal kriptografi kuantum), `initial_branches` bernilai `None` dan loop `submit` gagal dengan `TypeError: 'NoneType' object is not iterable`. Setelah placeholder diisi pun, setiap pekerja memanggil `pruning_tot(model, tokenizer, branch)`: teks cabang menjadi *soal baru*, dan bila cabang hanya berisi teks lanjutan, pertanyaan aslinya tidak ikut. Ketiga pekerja juga memakai satu model di satu perangkat.

Subbagian berikut mengukur apa yang sebenarnya mempercepat ekspansi, dan subbagian sesudahnya membangun versi BFS yang memperluas satu tingkat pohon dalam satu panggilan `generate`.

### Under the hood 4 (UTH-4): thread atau batch?

Enam ekspansi (ekspansi pertama keenam soal contoh buku, `UTH_TOKENS` token, greedy supaya hasilnya bisa dibandingkan) dikerjakan dengan tiga cara: berurutan, lewat `ThreadPoolExecutor(max_workers=3)` seperti `parallel_tot`, dan sebagai satu batch `generate`. Satu panggilan pemanasan mencegah cara pertama menanggung biaya awal seperti inisialisasi kernel; setiap waktu diambil setelah `sync()`, dan semuanya diulang dua kali.

In [21]:
@torch.no_grad()
def expand_one(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    out = model.generate(**inputs, max_new_tokens=UTH_TOKENS, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)


def timed(fn):
    sync()
    start = time.perf_counter()
    result = fn()
    sync()
    return result, time.perf_counter() - start


def run_threads():
    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as pool:
        return list(pool.map(expand_one, first_expansions))


def run_batch():
    return [node[0] for node in generate_batch(first_expansions, k=1, new_tokens=UTH_TOKENS, do_sample=False)]


expand_one(first_expansions[0])                          # pemanasan
for rep in (1, 2):
    sequential, t_seq = timed(lambda: [expand_one(p) for p in first_expansions])
    threaded, t_thr = timed(run_threads)
    batched, t_bat = timed(run_batch)
    same_thr = sum(a == b for a, b in zip(threaded, sequential))
    same_bat = sum(a == b for a, b in zip(batched, sequential))
    print(f"ulangan {rep}: berurutan {t_seq:.2f} s | 3 thread {t_thr:.2f} s ({t_seq / t_thr:.2f}x) | "
          f"satu batch {t_bat:.2f} s ({t_seq / t_bat:.2f}x) | teks sama: thread {same_thr}/6, batch {same_bat}/6")
for i, (b, s) in enumerate(zip(batched, sequential)):
    if b != s:
        print(f"soal {i + 1}: batch berpisah dari berurutan setelah {first_difference(b, s)} token yang sama | "
              f"awal batch {b[:50]!r} | awal berurutan {s[:50]!r}")

ulangan 1: berurutan 9.90 s | 3 thread 14.33 s (0.69x) | satu batch 2.10 s (4.72x) | teks sama: thread 6/6, batch 6/6
ulangan 2: berurutan 10.10 s | 3 thread 14.86 s (0.68x) | satu batch 2.09 s (4.83x) | teks sama: thread 6/6, batch 6/6


Angka dalam kurung adalah kecepatan relatif terhadap eksekusi berurutan (2,00x berarti separuh waktu). Di run sumber (CPU 2 vCPU), tiga thread hanya 1,02–1,09x dan satu batch 2,32–2,39x, dengan teks identik. Penjelasan yang paling mungkin: PyTorch sudah memakai kedua vCPU untuk setiap operasi, sehingga thread hanya berebut inti. Di CPU berinti lebih banyak, hasilnya bisa lain.

Di GPU kita mengharapkan batch tetap unggul: thread-thread memakai satu model di satu GPU sehingga kernel mereka mengantre, sedangkan batch membuat setiap kernel mengerjakan enam prompt sekaligus. Angka Anda yang menentukan apakah harapan itu, dan klaim buku tentang thread, berlaku di GPU. Angka "teks sama" untuk batch bisa kurang dari 6/6 di float16 atau bfloat16, karena bentuk tensor yang berbeda mengubah urutan penjumlahan; baris tambahan menunjukkan setelah berapa token pasangan yang berbeda berpisah.

### Versi GPU: BFS dengan satu batch per tingkat

Kalau percepatan memang datang dari batch, bentuk pencarian yang cocok untuk GPU adalah BFS: semua simpul satu kedalaman diperluas dalam satu panggilan `generate`. Fungsi berikut menuliskan setiap bagian ToT secara eksplisit:

- **state**: `Node` berisi teks jalur itu, skor cabang terakhir, dan kedalamannya;
- **pembangkitan**: `generate_batch` memperluas seluruh *frontier* (simpul yang menunggu diperluas) sekaligus, `branches` cabang per simpul;
- **penilaian**: log-prob rata-rata setiap cabang baru diberikan konteksnya, di GPU;
- **pemangkasan**: hanya `keep` cabang terbaik yang diteruskan (berbeda dari beam search UTH-1, yang bekerja per token di dalam `generate`);
- **berhenti**: pada `max_depth`, atau lebih awal bila tingkat berikutnya akan melampaui `token_budget`, yang dihitung dari batas atas `new_tokens` per cabang.

Penilainya sengaja log-prob, bukan penilai buku yang hampir selalu memberi 1,0. Ingat yang diukurnya: kewajaran teks menurut model, bukan kebenaran.

In [22]:
from dataclasses import dataclass


@dataclass
class Node:
    text: str = ""        # yang sudah ditulis model di jalur ini
    score: float = 0.0    # skor cabang terakhir
    depth: int = 0


def bfs_tot(problem, branches=3, keep=2, max_depth=3, new_tokens=NEW_TOKENS, token_budget=2000):
    root = tot_prompt(problem) + "\n\nLet's explore further:\n"
    frontier = [Node()]
    spent = 0
    for depth in range(max_depth):
        cost = len(frontier) * branches * new_tokens
        if spent + cost > token_budget:
            print(f"kedalaman {depth}: berhenti, butuh {cost} token lagi dari sisa anggaran {token_budget - spent}")
            break
        contexts = [root + node.text for node in frontier]
        expansions = generate_batch(contexts, k=branches, new_tokens=new_tokens, do_sample=True, top_k=50)
        spent += cost
        children, child_contexts, new_texts = [], [], []
        for node, context, texts in zip(frontier, contexts, expansions):
            for text in texts:
                children.append(Node(text=node.text + text, depth=depth + 1))
                child_contexts.append(context)
                new_texts.append(text)
        for child, score in zip(children, mean_logprob(child_contexts, new_texts, sep="")):
            child.score = float(score)
        children.sort(key=lambda n: n.score, reverse=True)
        frontier = children[:keep]                        # pangkas: sisanya tidak ditelusuri
        print(f"kedalaman {depth}: {len(contexts)} simpul -> {len(children)} cabang dalam 1 panggilan generate | "
              f"skor {' '.join(f'{c.score:.2f}' for c in children)} | diteruskan {len(frontier)}")
    return max(frontier, key=lambda n: n.score), spent

In [23]:
problem = "What are the potential implications of quantum computing on cryptography?"
torch.manual_seed(0)
start = time.perf_counter()
best, spent = bfs_tot(problem, max_depth=4, token_budget=20 * NEW_TOKENS)
sync()
print(f"\n{time.perf_counter() - start:.1f} s | anggaran terpakai {spent} token | "
      f"jalur terbaik (skor cabang terakhir {best.score:.2f}), 500 karakter pertama:")
print(best.text[:500])

kedalaman 0: 1 simpul -> 3 cabang dalam 1 panggilan generate | skor -1.33 -1.79 -2.23 | diteruskan 2
kedalaman 1: 2 simpul -> 6 cabang dalam 1 panggilan generate | skor -1.51 -2.05 -2.08 -2.34 -inf -inf | diteruskan 2
kedalaman 2: 2 simpul -> 6 cabang dalam 1 panggilan generate | skor -1.41 -1.59 -1.69 -1.88 -2.09 -2.11 | diteruskan 2
kedalaman 3: berhenti, butuh 600 token lagi dari sisa anggaran 500

9.7 s | anggaran terpakai 1500 token | jalur terbaik (skor cabang terakhir -1.41), 500 karakter pertama:
1) Path 1: First, we could...
2) Then, we might...
3) Finally, we'd need to...
As you can see from the above, the Tree-of-Thoughts approach is a straightforward one-liner. A bit more analysis is needed, however. Let me briefly explain all the different components. Here they are:
(1) First-ever quantum algorithm. We'll have a "quantum algorithm" that will allow us to break into any computer and, with time on our hands, figure out the encryption key and any other key we want. All at the 

Log per tingkat memperlihatkan seluruh siklus. Di GPU (`NEW_TOKENS` = 100, anggaran 2.000 token), tingkat pertama menghasilkan 3 cabang dari 1 simpul, tingkat kedua dan ketiga masing-masing 6 cabang dari 2 simpul, satu panggilan `generate` per tingkat; dua yang terbaik diteruskan. Tingkat keempat tidak dijalankan: tiga tingkat sudah memakai 300 + 600 + 600 = 1.500 token, dan tingkat berikutnya butuh 600 lagi. Dibandingkan `pruning_tot` (13 panggilan berurutan untuk 39 cabang), pohon ini memakai 3 panggilan untuk 15 cabang, dan setiap pangkasan benar-benar mengubah apa yang ditelusuri.

Mutu jawabannya tidak dijamin. Log-prob yang mendekati 0 sering datang dari pengulangan; kalau jalur terbaiknya berputar-putar di frasa yang sama, itulah pelajaran UTH-3 di dalam pencarian sungguhan.

## Arah ke depan: pohon dinamis

Sebelum ringkasan bab, buku membahas empat arah pengembangan: pohon dinamis, gabungan ToT dengan CoT, meta-learning untuk ToT, dan pengetahuan eksternal. Untuk yang pertama ada `dynamic_tot`: kompleksitas pikiran saat ini menentukan jumlah cabang, antara 2 dan 5, sehingga jalur yang rumit dijelajahi lebih lebar.

> **Listing buku: `dynamic_tot` tidak bisa dikompilasi**
>
> ```python
> def assess_complexity(thought):
>     # Implement logic to assess the complexity of the current thought
>     pass
> def determine_branches(complexity):
>     # Determine the number of branches based on complexity
>     return max(2, min(5, int(complexity 10)))
> ```
>
> `int(complexity 10)` kehilangan operator `*`, kemungkinan termakan saat pencetakan, sehingga sel ini ditolak dengan `SyntaxError` dan contoh pemakaiannya gagal dengan `NameError`. Setelah `*` dikembalikan, `assess_complexity` dan `generate_branches` masih `pass`, dan `None * 10` gagal dengan `TypeError`.

Versi berikut mengembalikan `*`, mengisi `assess_complexity` dengan keragaman leksikal (kata unik ÷ semua kata) dan `generate_branches` dengan sampling di perangkat model. Sel ini juga menghitung ukuran pohon pada kedalaman bawaan buku, 5, yang murni aritmetika; pohon yang benar-benar dijalankan dibatasi `DYN_DEPTH` tingkat (2 di GPU).

In [24]:
def dynamic_tot(model, tokenizer, problem, max_depth=5, new_tokens=NEW_TOKENS, trace=None):
    def adapt_structure(current_thought, depth):
        if depth == max_depth:
            return current_thought
        complexity = assess_complexity(current_thought)
        num_branches = determine_branches(complexity)
        if trace is not None:
            trace.append((depth, round(complexity, 3), num_branches))
        branches = generate_branches(
            model, tokenizer, current_thought, num_branches
        )
        results = []
        for branch in branches:
            results.append(
                adapt_structure(
                    current_thought + branch, depth + 1
                )
            )
        return max(results, key=lambda x: evaluate_thought(x))
    def assess_complexity(thought):
        words = thought.split()                              # diisi: keragaman leksikal, 0-1
        return len(set(words)) / len(words)
    def determine_branches(complexity):
        # Determine the number of branches based on complexity
        return max(2, min(5, int(complexity * 10)))          # perbaikan: '*' yang hilang
    def generate_branches(model, tokenizer, thought, num_branches):
        inputs = tokenizer(thought + "\n\nLet's explore further:\n", return_tensors="pt").to(model.device)
        outputs = model.generate(**inputs, max_new_tokens=new_tokens, num_return_sequences=num_branches,
                                 do_sample=True, pad_token_id=tokenizer.eos_token_id)
        return [tokenizer.decode(o[inputs["input_ids"].shape[1]:], skip_special_tokens=True) for o in outputs]
    initial_prompt = tot_prompt(problem)
    return adapt_structure(initial_prompt, 0)


print("ukuran pohon dynamic_tot pada max_depth=5 bila setiap simpul mendapat b cabang:")
for b in range(2, 6):
    calls = sum(b ** d for d in range(5))
    branches = sum(b ** d for d in range(1, 6))
    print(f"   b = {b}: {calls:>4} panggilan generate, {branches:>5} cabang")

ukuran pohon dynamic_tot pada max_depth=5 bila setiap simpul mendapat b cabang:
   b = 2:   31 panggilan generate,    62 cabang
   b = 3:  121 panggilan generate,   363 cabang
   b = 4:  341 panggilan generate,  1364 cabang
   b = 5:  781 panggilan generate,  3905 cabang


In [25]:
problem = "How might advancements in nanotechnology impact medicine in the next decade?"
trace = []
torch.manual_seed(0)
start = time.perf_counter()
solution = dynamic_tot(model, tokenizer, problem, max_depth=DYN_DEPTH, trace=trace)
sync()
print(f"max_depth={DYN_DEPTH}: {len(trace)} panggilan generate | {time.perf_counter() - start:.1f} s")
for depth, complexity, n in trace:
    print(f"   kedalaman {depth}: kompleksitas {complexity} -> {n} cabang")
print("skor evaluate_thought jawaban akhir:", evaluate_thought(solution))

max_depth=2: 6 panggilan generate | 16.9 s
   kedalaman 0: kompleksitas 0.61 -> 5 cabang
   kedalaman 1: kompleksitas 0.5 -> 5 cabang
   kedalaman 1: kompleksitas 0.553 -> 5 cabang
   kedalaman 1: kompleksitas 0.539 -> 5 cabang
   kedalaman 1: kompleksitas 0.462 -> 4 cabang
   kedalaman 1: kompleksitas 0.588 -> 5 cabang
skor evaluate_thought jawaban akhir: 1.0


Tabel ukuran menunjukkan mengapa "dinamis" perlu rem: pada kedalaman bawaan 5, pohonnya 31 panggilan `generate` bila setiap simpul mendapat 2 cabang dan 781 panggilan (3.905 cabang) bila mendapat 5, untuk satu soal. Akar soal nanoteknologi selalu mendapat lebar maksimum: keragaman leksikal template-nya 0,61, dan `int(0.61 * 10)` = 6 dipotong menjadi 5, sebelum model menulis apa pun. Di tingkat berikutnya kompleksitas dihitung ulang, jadi log menunjukkan apakah teks model mengubah lebar pohon.

`max()` di akhir tetap memakai `evaluate_thought`, jadi semua daun bernilai 1,0 dan daun pertama yang terpilih. Pohon dinamis menambah cara *mengatur lebar*, bukan cara *menilai*.

### Tiga arah lain dan catatan atas rujukannya

Untuk tiga arah lainnya buku hanya memberi deskripsi dan rujukan, dan beberapa perlu diluruskan (dari literatur, diperiksa terhadap makalah aslinya):

- **Gabungan ToT dan CoT.** arXiv 2409.17433 adalah HDFlow, tentang peralihan antara berpikir cepat dan lambat lewat *workflow* dinamis; abstraknya tidak menyebut ToT.
- **Pengetahuan eksternal.** arXiv 2407.00653 adalah Chain-of-Knowledge, tentang melatih LLM menalar dengan aturan dari *knowledge graph*; abstraknya juga tidak menyebut ToT.
- **Meta-learning untuk ToT.** Buku menyebutnya belum dieksplorasi. Long (2023) merancang pengendali ToT yang bisa dilatih sebagai *policy network* dengan REINFORCE (eksperimennya sendiri memakai pengendali berbasis aturan), dan ToTRL (Wu dkk., Mei 2025) melatih penalaran ToT dengan *reinforcement learning*. Yang terkait erat, *Chain of Preference Optimization* (Zhang dkk. 2024) melatih model dengan pilihan dari pencarian ToT yang dijalankannya sendiri, sehingga saat dipakai model cukup menulis satu rantai CoT tanpa membangun pohon.

## Memeriksa perangkat

Sel terakhir memeriksa bahwa pekerjaan berat memang terjadi di GPU. Puncak memori yang dialokasikan PyTorch seharusnya di atas ukuran bobot model (sekitar 1,44 GiB untuk `gpt2-large` di float16 atau bfloat16), karena *cache KV*, yaitu vektor key dan value token-token sebelumnya yang disimpan agar tidak dihitung ulang di setiap langkah generasi, juga tinggal di GPU.

In [26]:
print("parameter model di", sorted({str(p.device) for p in model.parameters()}), "| dtype", next(model.parameters()).dtype)
print(f"bobot model: {sum(p.numel() * p.element_size() for p in model.parameters()) / 2**30:.2f} GiB")
if DEVICE == "cuda":
    print(f"puncak memori GPU yang dialokasikan PyTorch: {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB "
          f"dari {vram_total / 2**30:.1f} GiB")

parameter model di ['cuda:0'] | dtype torch.float16
bobot model: 1.44 GiB
puncak memori GPU yang dialokasikan PyTorch: 3.52 GiB dari 14.6 GiB


## Ringkasan

Buku menutup bab dengan perbandingan: CoT untuk penalaran linear, ToT untuk masalah yang jalurnya perlu dibandingkan dan bisa buntu. Yang ditambahkan notebook ini:

- **Penilai memindahkan biaya, tidak menghapusnya.** Di Game of 24, pemangkasan menurunkan ekspansi DFS dari 157 ke 11, tetapi penilaiannya sendiri setara 162 ekspansi.
- **Listing buku butuh perbaikan agar berjalan**: sampling untuk cabang, `**inputs`, batas token, penilai yang terisi, operator `*`, dan placeholder.
- **Cabang yang beragam datang dari sampling**; di run sumber, beam search memberi cabang yang nyaris sama.
- **Penilai buku hanya menghitung kata**: template membuat semua skor 1,0, sehingga DFS dan pemangkasan selalu memilih cabang pertama; urutan kata tidak berpengaruh, dan pada rantai MultiArith benar lawan salah AUC-nya 0,527.
- **Manfaat bercabang milik penilai**: di tugas B run sumber, hanya pemeriksa yang mengukur tujuan secara langsung yang mencapai oracle; di tugas A hampir tidak ada kandidat yang benar, sehingga tidak ada penilai yang bisa menolong.
- **State harus dirancang**: soal bertahap baru muat di konteks 1.024 token setelah yang dibawa ke langkah berikutnya dibatasi pada 150 token terakhir.
- **Untuk biaya, batch lebih menjanjikan daripada thread**: di run sumber batch jauh lebih cepat, dan BFS dengan satu batch per tingkat memanfaatkannya.

Bab berikutnya di buku membahas ReAct (Yao dkk. 2022), yang menyelingi penalaran dengan tindakan, misalnya memanggil alat lalu membaca hasilnya. Pertanyaan untuk dibawa ke sana: kalau pemeriksa di luar model adalah satu-satunya penilai yang andal, apa yang terjadi bila model sendiri boleh memanggilnya?